# Zirconium loop line broadening derivation

This notebook derives a parameterized XRD line-broadening workflow for zirconium containing loop-driven anisotropic strain broadening.


## Section 1 — Scope and symbols

We model observed broadening as the combination of instrumental, size, and strain terms.

\[
\beta_{\mathrm{obs}}^2 \approx \beta_{\mathrm{inst}}^2 + \beta_{\mathrm{size}}^2 + \beta_{\mathrm{strain}}^2
\]

All zirconium-specific constants are external inputs and are **not** hard-coded.


In [ ]:
import numpy as np

beta_inst = 0.0018
beta_size = 0.0021
beta_strain = 0.0014
beta_obs = np.sqrt(beta_inst**2 + beta_size**2 + beta_strain**2)
beta_obs


## Section 2 — Required external inputs

| Input | Symbol | Meaning | Units | Source |
|---|---|---|---|---|
| X-ray wavelength | $\lambda$ | Probe wavelength | m | Instrument setup |
| Zirconium lattice parameters | $a, c$ | HCP cell parameters | m | Refined or literature |
| Contrast baseline | $\bar{C}$ | Average dislocation contrast | – | Elastic/dislocation model |
| Contrast anisotropy coefficient | $q$ | Reflection anisotropy term | – | Elastic/dislocation model |
| Burgers vector magnitude | $b$ | Loop/dislocation Burgers magnitude | m | Defect model |
| Dislocation/loop density | $\rho$ | Defect density | m$^{-2}$ | Fitted or independent estimate |
| Shape factor | $K$ | Scherrer constant | – | Profile model assumption |
| Instrument broadening coefficients | $U,V,W$ | Caglioti parameters | rad$^2$ | Instrument calibration |


## Section 3 — Reciprocal-space transform

\[
q = \frac{4\pi}{\lambda}\sin\theta,\qquad 2\theta = 2\arcsin\left(\frac{q\lambda}{4\pi}\right)
\]


In [ ]:
from src.xrd_line_broadening import two_theta_to_q, q_to_two_theta

wavelength = 1.5406e-10

two_theta = np.deg2rad(np.array([30.0, 40.0, 50.0]))
q = two_theta_to_q(two_theta, wavelength)
two_theta_back = q_to_two_theta(q, wavelength)
q, np.rad2deg(two_theta_back)


## Section 4 — Size broadening (Scherrer)

\[
\beta_{\mathrm{size}} = \frac{K\lambda}{L\cos\theta}
\]


In [ ]:
from src.xrd_line_broadening import scherrer_fwhm

K = 0.9
L = 80e-9
theta = np.deg2rad(np.array([15.0, 20.0, 25.0]))
beta_size = scherrer_fwhm(theta, wavelength=wavelength, domain_size=L, k=K)
beta_size


## Section 5 — Strain broadening and contrast factors

Isotropic microstrain form:
\[
\beta_{\mathrm{strain}} = 4\varepsilon\tan\theta
\]

Reflection anisotropy form:
\[
C_{hkl}=\bar{C}\left(1-q\,\eta_{hkl}\right),\qquad
\eta_{hkl}=\frac{\frac{4}{3}(h^2+hk+k^2)}{\frac{4}{3}(h^2+hk+k^2)+(lc/a)^2}
\]


In [ ]:
from src.xrd_line_broadening import microstrain_fwhm
from src.contrast_factors import HKL, contrast_factor_hex

epsilon = 8e-4
beta_strain = microstrain_fwhm(theta, microstrain_rms=epsilon)

c_over_a = 1.593  # replace with zirconium value from external source
c_bar = 0.28      # external input
q_aniso = 1.6     # external input
hkl = HKL(1, 0, 1)
C_hkl = contrast_factor_hex(hkl.h, hkl.k, hkl.l, c_over_a=c_over_a, c_bar=c_bar, q=q_aniso)

beta_strain, C_hkl


## Section 6 — Instrumental broadening model

\[
\beta_{\mathrm{inst}}^2 = U\tan^2\theta + V\tan\theta + W
\]


In [ ]:
from src.xrd_line_broadening import caglioti_fwhm

U, V, W = 1.0e-5, 2.0e-5, 4.0e-5
beta_inst = caglioti_fwhm(theta, U, V, W)
beta_inst


## Section 7 — Composite width equations

For Gaussian-like components:
\[
\beta_{\mathrm{obs}} = \sqrt{\beta_{\mathrm{inst}}^2 + \beta_{\mathrm{size}}^2 + \beta_{\mathrm{strain}}^2}
\]

For Lorentzian-like components:
\[
\beta_{\mathrm{obs}} = \beta_{\mathrm{inst}} + \beta_{\mathrm{size}} + \beta_{\mathrm{strain}}
\]


In [ ]:
from src.xrd_line_broadening import combine_fwhm_gaussian, combine_fwhm_lorentzian

beta_obs_g = combine_fwhm_gaussian(beta_inst, beta_size, beta_strain)
beta_obs_l = combine_fwhm_lorentzian(beta_inst, beta_size, beta_strain)
beta_obs_g, beta_obs_l


## Section 8 — Two-population a/c loop contribution model

Let $a$-type and $c$-type loop populations have weights $w_a,w_c$ and widths
$\beta_a,\beta_c$.

\[
\beta_{\mathrm{mix}} = \frac{w_a}{w_a+w_c}\beta_a + \frac{w_c}{w_a+w_c}\beta_c
\]


In [ ]:
from src.diffraction_simulation import two_population_ac_broadening

size_a = beta_size * 1.10
strain_a = beta_strain * 0.90
size_c = beta_size * 0.95
strain_c = beta_strain * 1.25

beta_mix = two_population_ac_broadening(
    theta_rad=theta,
    instrument_fwhm_rad=beta_inst,
    size_a=size_a,
    strain_a=strain_a,
    size_c=size_c,
    strain_c=strain_c,
    weight_a=0.65,
    weight_c=0.35,
)

beta_mix


## Section 9 — Synthetic profile construction and execution checklist

Peak profile (pseudo-Voigt):
\[
I(2\theta)=\eta I_L(2\theta)+(1-\eta)I_G(2\theta)
\]

Practical execution order:
1. Enter all required external zirconium inputs.
2. Compute reflection-wise $C_{hkl}$.
3. Build size and strain widths for each reflection.
4. Combine with instrumental broadening.
5. Simulate/fitted synthetic peaks for interpretation.


In [ ]:
from src.diffraction_simulation import build_population_peaks, synthetic_pattern

x = np.deg2rad(np.linspace(25, 65, 2000))
centers = np.deg2rad(np.array([30.0, 36.0, 48.0, 57.5]))
amplitudes = np.array([1200.0, 900.0, 1400.0, 850.0])
widths = np.interp(centers, np.linspace(theta.min(), theta.max(), len(beta_mix)), beta_mix)
peaks = build_population_peaks(centers, amplitudes, widths)
pattern = synthetic_pattern(x, peaks, background=45.0)

pattern[:5], pattern.max()
